In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/IMDB Dataset.csv")

In [3]:
df.drop_duplicates(inplace = True)

In [4]:
df.shape

(49582, 2)

# Preprocessing

# 1. Converting to Lowercase

In [5]:
df["review"] = df["review"].str.lower()

# 2. Removing the URLs

In [6]:
import re       

def remove_urls(text):
    text = re.sub(r"http\S+" , "" , text) # re.sub(pattern , replace , string)
    return text
df["review"] = df["review"].apply(remove_urls)

# docs.python.org/3/library/re.html#

# 3. Removing Punctuations

In [7]:
def remove_punct(text):
    text = re.sub(r"[^A-Za-z0-9\s]" , "" , text) 
    return text
df["review"] = df["review"].apply(remove_punct)

# 4. Removing HTML

In [8]:
def remove_html(text):
    text = re.sub(r"<.*?>" , "" , text) # "." -> Any character , * -> 0 or more character
    return text
df["review"] = df["review"].apply(remove_html)

# 5. Removing the StopWords

In [9]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\yamin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\yamin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\yamin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [10]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [11]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word , "")

    return text

df["review"] = df["review"].apply(remove_stopwords)

In [12]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


# 6. Stemming

In [13]:
from nltk.stem import PorterStemmer

def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [14]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


# 7. Encoding

In [19]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])

In [20]:
y = df["sentiment"]

# 8. Vectorization

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features = 5000)
X = tf.fit_transform(df["review"])

# Dataset & Dataloader

In [22]:
from sklearn.model_selection import train_test_split

X_train , X_test , y_train , y_test = train_test_split(
    X , y , test_size = 0.2 , random_state = 42
)

In [23]:
import torch 
from torch.utils.data import TensorDataset , DataLoader

In [24]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [25]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float() ,
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float() ,
    torch.from_numpy(y_test.values).float()
)

In [26]:
train_loader = DataLoader(train_set , shuffle=True , batch_size = 64)
test_loader = DataLoader(test_set , shuffle=True , batch_size = 64)

# Build the RNN

In [27]:
import torch.nn as nn
import torch.optim as optim

In [28]:
class RNN(nn.Module):
    def __init__(self , input_size , hidden_size = 128 , num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layers
        self.rnn = nn.RNN(input_size , hidden_size ,num_layers , batch_first = True) 
        ## batch-first = true => input shape becomes (batch , seq_len , features)
        ## input-size => number of features per timestep

        # fully-connected layer
        self.fc = nn.Linear(hidden_size , 1)

    def forward(self , x):
        # optional 
        h0 = torch.zeros(self.num_layers , x.size(0) , self.hidden_size)

        out , _ = self.rnn(x , h0)
        # 1st output = hidden state of all timesteps => (batch , seq_len , hidden_size)
        # 2nd output = final hidden state of last timesteps

        out = self.fc(out[: , -1 , :]) # Selects the last timestep output.
        return out

In [29]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss() # binary cross entrophy loss
optimizer = optim.Adam(model.parameters())

# Training the RNN

In [30]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb , yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction

        outputs = model(Xb) # (batch_size , 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size) => probability

        loss = criterion(outputs , yb)
        loss.backward()
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.28742894530296326
epoch = 2/10 and loss = 0.2424214482307434
epoch = 3/10 and loss = 0.21206887066364288
epoch = 4/10 and loss = 0.3768022954463959
epoch = 5/10 and loss = 0.23991578817367554
epoch = 6/10 and loss = 0.27930575609207153
epoch = 7/10 and loss = 0.204951211810112
epoch = 8/10 and loss = 0.20312385261058807
epoch = 9/10 and loss = 0.11891542375087738
epoch = 10/10 and loss = 0.2623331844806671


# Evaluation

In [32]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    total_vals = 0

    for Xb , yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        total_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"Accuracy = {correct_vals/total_vals * 100}")

Accuracy = 85.61056771200968
